# Cornell Pet Health RAG — 처음부터 관찰하기

이 노트북은 코드를 외우는 자료가 아니라, RAG의 각 단계를 눈으로 확인하는 실습입니다. 실제 로직은 `petcare_rag` 패키지에 있고 여기서는 그 함수를 호출합니다.

```text
한국어 질문 + dog/cat → 질문 임베딩 → Chroma 검색 → SOURCE 조립 → Gemini 답변 → Cornell 인용
```

> 범위: Cornell 일반 건강정보 설명. 확정 진단, 처방, 개인 위험도·응급 판단은 이 RAG의 역할이 아닙니다.

## 0. 시작 전 준비

PowerShell에서 `$env:GEMINI_API_KEY="발급받은_키"`를 설정한 뒤 이 노트북을 엽니다. API 키 자체는 화면에 출력하지 않습니다.

In [ ]:
import json, os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from petcare_rag import (
    build_context, build_generation_prompt, embed_question,
    open_collection, retrieve, run_pipeline,
)

DB_PATH = ROOT / 'rag_data' / 'chroma'
GOLD_PATH = ROOT / 'rag_data' / 'evaluation' / 'cornell_retrieval_gold.jsonl'
collection = open_collection(DB_PATH)
print('API 키 설정:', '완료' if os.environ.get('GEMINI_API_KEY') else '안 됨')
print('Chroma 청크:', collection.count())
print('컬렉션:', collection.name)

## 1. 질문도 숫자 좌표로 바뀐다

임베딩은 문장의 의미를 나타내는 숫자 배열입니다. 배열 전체는 길기 때문에 차원과 앞의 숫자 5개만 확인합니다. 숫자 하나하나를 사람이 해석할 필요는 없습니다.

In [ ]:
question = '강아지가 초콜릿을 먹으면 왜 위험한가?'
species = 'dog'
vector = embed_question(question)
print('질문:', question)
print('벡터 차원:', len(vector))
print('앞의 숫자 5개:', [round(x, 5) for x in vector[:5]])

## 2. species 필터가 왜 필요한가

아래의 `unfiltered`는 학습용 비교입니다. 실제 파이프라인의 `retrieve()`는 항상 species 필터를 적용합니다.

In [ ]:
unfiltered = collection.query(
    query_embeddings=[vector], n_results=5,
    include=['metadatas', 'distances'],
)
print('필터 없는 결과:')
for meta, distance in zip(unfiltered['metadatas'][0], unfiltered['distances'][0]):
    print(f"  {meta['species']} | {meta['title']} | similarity={1-distance:.4f}")

chunks = retrieve(question, species, top_k=5, collection=collection)
print('\ndog 필터 결과:')
for c in chunks:
    print(f"  {c.species} | {c.title} | similarity={c.similarity:.4f}")

## 3. top-k는 정답 개수가 아니다

`k`는 답변 재료로 가져올 최대 청크 수입니다. 너무 작으면 근거를 놓치고, 너무 크면 관련 없는 내용이 섞일 수 있습니다. 같은 검색 순위의 앞 1개·3개·5개를 비교합니다.

In [ ]:
for k in (1, 3, 5):
    print(f'\nTOP-{k}')
    for c in chunks[:k]:
        print(f'  {c.title} > {c.section_path[-1]}')

## 4. 검색 결과를 SOURCE 카드로 조립한다

Gemini가 어느 문장을 어느 출처에서 읽었는지 구분하도록 각 청크에 번호를 붙입니다. 최종 인용 `[1]`은 이 번호를 뜻합니다.

In [ ]:
context = build_context(chunks)
print(context)

## 5. Gemini가 받는 질문 프롬프트

시스템 규칙은 `petcare_rag.pipeline.SYSTEM_INSTRUCTION`에 있습니다. 아래는 사용자의 질문과 SOURCE가 합쳐진 부분입니다.

In [ ]:
generation_prompt = build_generation_prompt(question, species, context)
print(generation_prompt)

## 6. 전체 RAG 실행: 검색 → 답변 → 안전한 출처

Gemini가 적은 URL은 사용하지 않습니다. 프로그램이 검증된 SOURCE 번호를 Chroma 메타데이터의 Cornell URL로 변환합니다.

In [ ]:
response, trace = run_pipeline(question, species, top_k=5, collection=collection)
print(response.answer)
print('\n인용:')
for citation in response.citations:
    print(f'[{citation.number}] {citation.title} — {citation.url}')
print('\n안내:', response.disclaimer)

## 7. 근거가 부족한 질문 실험

벡터 검색은 어떤 질문에도 가까운 청크를 반환할 수 있습니다. 따라서 LLM이 SOURCE로 답할 수 없는 질문은 `insufficient_evidence=true`로 처리해야 합니다.

In [ ]:
irrelevant, _ = run_pipeline(
    '강아지가 올해 소득세 신고를 하려면 어떤 서류가 필요한가?',
    'dog', top_k=5, collection=collection,
)
print('근거 부족:', irrelevant.insufficient_evidence)
print(irrelevant.answer)
print('인용 수:', len(irrelevant.citations))

## 8. 종을 잘못 지정하면 어떻게 되는가

질문 문장보다 `species` 필터가 우선입니다. 실제 앱에서는 사용자가 임의로 고르기보다 반려동물 프로필의 species를 전달해야 합니다.

In [ ]:
wrong_species = retrieve(question, 'cat', top_k=5, collection=collection)
for c in wrong_species:
    print(c.species, c.title)
assert all(c.species == ['cat'] for c in wrong_species)

## 9. 골든 질문 12개로 검색 평가

아래 셀은 질문마다 임베딩 API를 호출합니다. 무료 할당량과 속도 제한을 확인한 뒤 실행하세요. 통과 조건은 기대 문서 중 하나가 top-5에 포함되는 것입니다.

In [ ]:
gold = [json.loads(line) for line in GOLD_PATH.open(encoding='utf-8')]
passed = 0
for case in gold:
    found = retrieve(case['query'], case['species'], case['top_k'], collection=collection)
    ids = {c.document_id for c in found}
    ok = bool(ids.intersection(case['expected_document_ids']))
    passed += ok
    print('PASS' if ok else 'FAIL', case['case_id'])
print(f'결과: {passed}/{len(gold)}')

## 10. 개선은 하나씩 비교한다

기준선이 정상 동작한 뒤에만 다음 순서로 실험합니다.

1. top-k 3·5·8 비교
2. 같은 문서 청크 중복 제한
3. 한국어 질문을 영어 검색문으로 변환
4. 한국어+영어 multi-query
5. 영어 검색문 기반 BM25 하이브리드
6. 다국어 cross-encoder 리랭커

한 번에 여러 방법을 켜면 어떤 변경이 좋아졌는지 알 수 없습니다. 항상 같은 골든 질문의 변경 전후 결과를 기록하세요.